# Step 6: Pipeline Orchestration with Dagster — Olist Customer Analytics Project

**Owner:** Step 6 / Pipeline Orchestration  
**Tooling:** Dagster + Meltano + dbt + BigQuery  
**Business Problem:** *Which customers are most likely to become repeat buyers?*

This notebook documents the optional Step 6 implementation for orchestrating the Olist ELT pipeline using Dagster.

The purpose of this step is to show how the separate pipeline components can be executed in sequence:

```text
Meltano ingestion
        ↓
dbt transformations
        ↓
dbt tests
        ↓
BigQuery analytics-ready tables
```

Dagster is used as the orchestration layer to coordinate, monitor, and optionally schedule the pipeline.


# 1. Business Context

The Olist project requires an end-to-end data pipeline that supports customer analytics and repeat-buyer segmentation.

The earlier pipeline phases cover:

- **Step 1:** Raw data ingestion into BigQuery using Meltano
- **Step 2:** Data warehouse design using a star schema
- **Step 3:** ELT transformations using dbt
- **Step 4:** Data quality testing
- **Step 5:** Data analysis with Python

Step 6 adds orchestration so the workflow can be executed and monitored from a single place.


# 2. Orchestration Objective

The objective of Dagster orchestration is to make the pipeline easier to operate.

Instead of manually running each command separately:

```bash
cd meltano-olist
meltano run tap-spreadsheets-anywhere target-bigquery

cd ../dbt_olist
dbt run
dbt test
```

Dagster defines these as ordered assets:

```text
meltano_ingestion → dbt_run → dbt_test
```

This allows the team to:

- Run the full workflow from the Dagster UI
- Track success and failure of each pipeline step
- View logs for each step
- Add scheduling later if automated refreshes are required


# 3. Project Folder Structure

The Dagster implementation is placed in a separate orchestration folder:

```text
olist-data-pipeline/
│
├── meltano-olist/
│   └── meltano.yml
│
├── dbt_olist/
│   ├── dbt_project.yml
│   └── models/
│
├── dagster-olist/
│   ├── dagster_olist/
│   │   ├── __init__.py
│   │   ├── assets.py
│   │   └── definitions.py
│   └── pyproject.toml
│
└── README.md
```

This keeps orchestration logic separate from ingestion logic and dbt transformation logic.


# 4. Environment Setup

The project was run from a conda environment named `elt`.

```bash
conda activate elt
```

The required Dagster packages are:

```bash
pip install dagster dagster-webserver dagster-dbt
```

The pipeline also requires Meltano and dbt to be available in the same environment:

```bash
pip install meltano dbt-core dbt-bigquery
```

## Why this matters

Dagster launches subprocesses to run Meltano and dbt commands. Therefore, the environment used to start Dagster must also be able to access:

- `meltano`
- `dbt`
- BigQuery credentials
- Python dependencies


In [ ]:
# Terminal command, shown here for documentation
# conda activate elt

# Install orchestration dependencies if needed
# pip install dagster dagster-webserver dagster-dbt

# Check installed command-line tools
# which dagster
# which meltano
# which dbt

# 5. Dagster Asset Definition

Dagster assets represent executable pipeline steps.

For this project, three assets are defined:

| Asset | Purpose |
|---|---|
| `meltano_ingestion` | Loads raw CSV data into BigQuery raw tables |
| `dbt_run` | Builds staging, intermediate, dimension, fact, and RFM models |
| `dbt_test` | Runs dbt data quality tests |

The assets are linked using dependencies so they run in the correct order.


In [ ]:
# File: dagster-olist/dagster_olist/assets.py

from pathlib import Path
import subprocess
from dagster import asset


PROJECT_ROOT = Path(__file__).resolve().parents[2]
MELTANO_DIR = PROJECT_ROOT / "meltano-olist"
DBT_DIR = PROJECT_ROOT / "dbt_olist"


@asset
def meltano_ingestion():
    """Run Meltano ingestion to load raw Olist CSV files into BigQuery."""
    subprocess.run(
        ["meltano", "run", "tap-spreadsheets-anywhere", "target-bigquery"],
        cwd=MELTANO_DIR,
        check=True,
    )


@asset(deps=[meltano_ingestion])
def dbt_run():
    """Run dbt transformations to build warehouse models in BigQuery."""
    subprocess.run(
        ["dbt", "run"],
        cwd=DBT_DIR,
        check=True,
    )


@asset(deps=[dbt_run])
def dbt_test():
    """Run dbt data quality tests after transformations complete."""
    subprocess.run(
        ["dbt", "test"],
        cwd=DBT_DIR,
        check=True,
    )


## 5.1 Important Implementation Detail

The `cwd` argument is important.

Meltano must be executed inside the `meltano-olist/` folder because this is where `meltano.yml` lives.

dbt must be executed inside the `dbt_olist/` folder because this is where `dbt_project.yml` lives.

Without setting `cwd`, Dagster may run the commands from the `dagster-olist/` folder and produce errors such as:

```text
Error: `meltano run` must be run inside a Meltano project.
```


# 6. Dagster Definitions

The `Definitions` object tells Dagster which assets belong to this project.

Dagster looks for a variable named `defs`.


In [ ]:
# File: dagster-olist/dagster_olist/definitions.py

from dagster import Definitions
from .assets import meltano_ingestion, dbt_run, dbt_test


defs = Definitions(
    assets=[
        meltano_ingestion,
        dbt_run,
        dbt_test,
    ]
)


# 7. pyproject.toml Configuration

To allow Dagster to discover the project automatically, add this file:

```toml
# File: dagster-olist/pyproject.toml

[tool.dagster]
module_name = "dagster_olist.definitions"
```

This allows the following command to work from inside `dagster-olist/`:

```bash
dagster dev
```

Alternatively, the module can be passed directly:

```bash
dagster dev -m dagster_olist.definitions
```


# 8. Running Dagster Locally

From the project root:

```bash
cd dagster-olist
dagster dev
```

Expected log output:

```text
Serving dagster-webserver on http://127.0.0.1:3000
```

Then open:

```text
http://127.0.0.1:3000
```

From the Dagster UI, materialize the assets to run the pipeline.


In [ ]:
# Terminal command, shown here for documentation

# cd dagster-olist
# dagster dev

# Open in browser:
# http://127.0.0.1:3000

# 9. Pipeline Execution Flow

When the assets are materialized, Dagster executes the pipeline in this order:

```text
1. meltano_ingestion
   └── Runs Meltano ingestion into BigQuery raw tables

2. dbt_run
   └── Builds staging, intermediate, fact, dimension, and RFM models

3. dbt_test
   └── Runs dbt tests to validate model quality
```

Dagster captures logs for each step and marks each asset as successful or failed.


# 10. Manual vs Automated Orchestration

At this stage, the pipeline is manually triggered from the Dagster UI.

This means:

```text
User clicks Materialize
        ↓
Dagster runs the workflow
        ↓
Pipeline steps execute in dependency order
```

This still counts as orchestration because Dagster coordinates and monitors the workflow.

A schedule can be added later to make it fully automated.


# 11. Optional Daily Schedule

The following schedule can be added if the team wants the pipeline to run automatically every day at 9:00 AM.

```python
from dagster import define_asset_job, ScheduleDefinition


olist_pipeline_job = define_asset_job(
    name="olist_pipeline_job",
    selection=[
        "meltano_ingestion",
        "dbt_run",
        "dbt_test",
    ],
)


daily_olist_schedule = ScheduleDefinition(
    job=olist_pipeline_job,
    cron_schedule="0 9 * * *",
)
```

This converts the pipeline from manually triggered orchestration to scheduled orchestration.


# 12. Evidence of Successful Orchestration

The following log messages show Dagster operating correctly:

```text
Serving dagster-webserver on http://127.0.0.1:3000
```

This confirms that the Dagster UI is running.

```text
RUN_START - Started execution of run for "__ASSET_JOB"
```

This confirms that a Dagster run was triggered.

```text
STEP_START - Started execution of step "meltano_ingestion"
```

This confirms that Dagster started executing the first pipeline asset.

```text
Environment 'dev' is active
Installing extractor 'tap-spreadsheets-anywhere'
Installing loader 'target-bigquery'
```

This confirms that Dagster successfully called Meltano.


# 13. Error Handling Observed

During implementation, the following error appeared:

```text
Error: `meltano run` must be run inside a Meltano project.
```

## Root cause

Dagster was initially executing the Meltano command from the wrong folder.

## Fix

The asset was updated to use:

```python
cwd=MELTANO_DIR
```

This ensures Meltano runs inside the `meltano-olist/` project folder where `meltano.yml` is located.


# 14. Business and Technical Value

## Business value

Dagster helps the team operationalize the Olist analytics pipeline. Instead of relying on manual command execution, the pipeline can be run and monitored through a single orchestration interface.

This supports:

- More reliable data refreshes
- Faster issue diagnosis
- Better visibility for technical stakeholders
- Repeatable pipeline execution

## Technical value

Dagster provides:

- Asset dependency management
- Logs for each execution step
- Clear failure points
- A UI for monitoring
- Scheduling support
- Extensibility for future production workflows


# 15. Step 6 Completion Summary

Step 6 is considered implemented because the project now includes a Dagster orchestration layer that can execute the ELT workflow in sequence.

## Completed components

- Dagster project folder created
- Dagster assets defined
- Meltano ingestion asset added
- dbt run asset added
- dbt test asset added
- Asset dependencies configured
- Dagster UI launched successfully
- Pipeline run triggered from Dagster UI
- Logs captured by Dagster

## Current status

```text
Manual trigger through Dagster UI
        ↓
Automated execution of pipeline steps
```

## Future enhancement

Add a Dagster schedule to run the pipeline automatically on a daily or weekly basis.


# 16. Final Architecture

```text
Raw Olist CSV Files
        ↓
Meltano
        ↓
BigQuery Raw Dataset: olist_raw
        ↓
dbt Staging Models
        ↓
dbt Intermediate Models
        ↓
dbt Mart Models / Star Schema
        ↓
Customer RFM Table
        ↓
Python Analysis / Executive Reporting
```

Dagster sits above the pipeline and coordinates the execution:

```text
Dagster
  ├── meltano_ingestion
  ├── dbt_run
  └── dbt_test
```


# 17. Report Text for Technical Documentation

The following paragraph can be reused in the technical report:

> Dagster was implemented as the orchestration layer for the Olist ELT pipeline. The pipeline was represented as three ordered assets: `meltano_ingestion`, `dbt_run`, and `dbt_test`. The first asset executes the Meltano ingestion process to load raw Olist CSV files into BigQuery. The second asset runs dbt transformations to build staging, intermediate, dimension, fact, and RFM models in the data warehouse. The final asset runs dbt tests to validate the transformed data. Dagster provides a central UI for triggering the pipeline, monitoring execution status, reviewing logs, and identifying failures. This improves pipeline reliability and makes the workflow easier to operate and extend.
